In [21]:
from pathlib import Path
import json
import regex
import time

In [22]:
REGEX_PATTERN = r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"

In [23]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

# The stats
What we are trying to have is by key of token id, we count

## Freqeuncy Count
Frequency of occurence, we'll count the early layers of occurance first then later

like if ABCD is a token, we add count to all the following tokens that represent: A, AB, B, ABC, C, ABCD, D

## Integrity
Is this token form at least a presentable charactor,

True: like presentable punctuation, chinese charactor etc

False: like the bytes in half a Chinese charactor, will throw error if decode.

## The Phrase
If above is True

The text, word, phrase the token is reprensenting

So result format should look like
```json
{
    2500: {
        "ct": 20350,
        "integrity": true,
        "text": "花生"
    }
}
```

or 
```json
{
    500: {
        "ct": 123456,
        "integrity": false
    }
}
```


In [24]:
from tqdm.auto import tqdm

In [32]:
def run_stats(vocab, text):
    pat = regex.compile(REGEX_PATTERN)
    pair_to_code = {(v[0], v[1]): k for k, v in vocab.items()}

    def encode_segment(segment):
        ids = list(segment.encode("utf8"))
        if len(ids) < 2:
            return ids
        while len(ids) > 1:
            temp_vocab = {}
            for c1, c2 in zip(ids, ids[1:]):
                token = pair_to_code.get((c1, c2))
                if token is not None:
                    temp_vocab[(c1, c2)] = token
            if not temp_vocab:
                break
            pair = min(temp_vocab, key=lambda x: temp_vocab[x])
            token = temp_vocab[pair]
            new_ids = []
            i = 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                    new_ids.append(token)
                    i += 2
                else:
                    new_ids.append(ids[i])
                    i += 1
            ids = new_ids
        return ids

    subtokens_cache = {}

    def get_subtokens(token_id):
        if token_id in subtokens_cache:
            return subtokens_cache[token_id]
        if token_id not in vocab:
            result = [token_id]
        else:
            c1, c2 = vocab[token_id]
            result = [token_id] + get_subtokens(c1) + get_subtokens(c2)
        subtokens_cache[token_id] = result
        return result

    freq = {}
    for segment in pat.findall(text):
        for token_id in encode_segment(segment):
            for sub in get_subtokens(token_id):
                freq[sub] = freq.get(sub, 0) + 1

    bytes_cache = {}

    def token_bytes(token_id):
        if token_id in bytes_cache:
            return bytes_cache[token_id]
        if token_id not in vocab:
            result = bytes([token_id])
        else:
            c1, c2 = vocab[token_id]
            result = token_bytes(c1) + token_bytes(c2)
        bytes_cache[token_id] = result
        return result

    stats = {}
    for token_id in list(range(256)) + list(vocab.keys()):
        try:
            text_repr = token_bytes(int(token_id)).decode("utf8")
            stats[int(token_id)] = {"ct": freq.get(int(token_id), 0), "integrity": True, "text": text_repr}
        except UnicodeDecodeError:
            stats[int(token_id)] = {"ct": freq.get(int(token_id), 0), "integrity": False}

    return stats

In [33]:
def tag_to_text(tag):
    CORPUS_DATA = get_locations(tag) 
    if tag == "jy":
        text = "\n".join(list(f.read_text() for f  in (CORPUS_DATA.parent.parent / tag).rglob("*.txt")))
    else:
        text = (CORPUS_DATA / "corpus.txt").read_text()

    return text


def tag_to_vocab(tag):
    CORPUS_DATA = get_locations(tag) 
    with open(CORPUS_DATA / "vocab.json") as f:
        vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())
    return vocab

In [34]:
for tag in tqdm(["xyj", "wm", "rtk", "drc", "jy"]):
    CORPUS_DATA = get_locations(tag) 
    text = tag_to_text(tag)
    vocab = tag_to_vocab(tag)
    stats = run_stats(vocab, text)
    with open(CORPUS_DATA / "stats.json", 'w') as f:
        f.write(json.dumps(stats, indent=2))

  0%|          | 0/5 [00:00<?, ?it/s]